In [0]:
%sql
--Corregimos datos para calcular estadisticas 
CREATE OR REPLACE VIEW estadisticas_clean AS
SELECT

  CASE 
    WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double
    ELSE 0
  END AS precio,

  moneda,

  CASE
    WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::DOUBLE
    ELSE 0
  END AS ambientes,

  CASE
    WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales ::DOUBLE
    ELSE 0
  END AS metros_cuadrados_totales,

  CASE
    WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos ::DOUBLE
    ELSE 0
  END AS metros_cuadrados_cubiertos,

  CASE
    WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::DOUBLE
    ELSE 0
  END AS antiguedad,

  tipo_de_operacion

FROM bootcamp_matias.raw.propiedades_bronze


In [0]:
%sql
--Estadistica de precio por moneda
SELECT
  moneda,
  COUNT(*) AS cantidad,
  ROUND(MIN(precio),2) AS precio_minimo,
  ROUND(MAX(precio),2) AS precio_maximo,
  ROUND(AVG(precio),2) AS precio_promedio,
  ROUND(PERCENTILE(precio,0.5),2) AS mediana,
  ROUND(PERCENTILE(precio,0.25),2) AS precio_percentil_25,
  ROUND(PERCENTILE(precio,0.75),2) AS precio_percentil_75
FROM estadisticas_clean
WHERE 
  precio IS NOT NULL 
  AND precio > 0
  AND moneda IN ('USD', 'ARS')
  AND moneda IS NOT NULL
GROUP BY 
  moneda



/*
Analisis de precio ARS:
La media (1.03M) es mucho mayor que la mediana (550k).
Esto indica una distribución muy sesgada a la derecha (outliers muy altos).

El 50% de las propiedades está entre:
420.000 ARS y 750.000 ARS

El máximo extremadamente alto (1.4M)
--------------------------------------------------------------------------------
Analisis de precio USD:
La media (169k) también es mayor que la mediana (100k).

Nuevamente hay sesgo positivo por propiedades muy caras.

El 50% del mercado está entre:
54.000 USD y 196.000 USD

Los valores 0.06 USD y 135M USD son outliers evidentes.


*/


In [0]:
%sql
--Estadsiticas de superficies m2_totales / m2_cubiertos
SELECT
  'm2_totales' AS columna,
  COUNT(*) AS cantidad,
  ROUND(MIN(metros_cuadrados_totales),2) AS minimo,
  ROUND(MAX(metros_cuadrados_totales),2) AS maximo,
  ROUND(AVG(metros_cuadrados_totales),2) AS promedio,
  ROUND(PERCENTILE(metros_cuadrados_totales,0.5),2) AS mediana,
  ROUND(PERCENTILE(metros_cuadrados_totales,0.25),2) AS percentil_25,
  ROUND(PERCENTILE(metros_cuadrados_totales,0.75),2) AS percentil_75
FROM estadisticas_clean
WHERE 
  metros_cuadrados_totales IS NOT NULL 
  AND metros_cuadrados_totales > 0

UNION ALL 

SELECT
  'm2_cubiertos' AS columna,
  COUNT(*) AS cantidad,
  ROUND(MIN(metros_cuadrados_cubiertos),2) AS minimo,
  ROUND(MAX(metros_cuadrados_cubiertos),2) AS maximo,
  ROUND(AVG(metros_cuadrados_cubiertos),2) AS promedio,
  ROUND(PERCENTILE(metros_cuadrados_cubiertos,0.5),2) AS mediana,
  ROUND(PERCENTILE(metros_cuadrados_cubiertos,0.25),2) AS percentil_25,
  ROUND(PERCENTILE(metros_cuadrados_cubiertos,0.75),2) AS percentil_75
FROM estadisticas_clean
WHERE 
  metros_cuadrados_cubiertos IS NOT NULL 
  AND metros_cuadrados_cubiertos > 0

/*
La superficie típica del dataset es ≈70 m².
El 50% de las propiedades tiene entre 45 y 170 m².
Existen outliers extremadamente grandes debido a errores de datos.
El promedio está distorsionado por valores inválidos.
Se recomienda filtrar valores mayores a 10.000 m² o usar percentiles para análisis.
*/

In [0]:
%sql
--Anlisis de antiguedad, posibles valores placeholder
SELECT
  antiguedad,
  COUNT(*) AS frecuencia,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(),2) AS porcentaje
FROM estadisticas_clean
GROUP BY 
  antiguedad
ORDER BY
  frecuencia DESC

/*
La variable antigüedad presenta un alto nivel de datos faltantes.
El valor 999 representa posiblemente un valor placeholder
Existen valores inválidos.
Se recomienda reemplazar 999 por NULL y filtrar valores negativos antes de usar la variable en análisis.
*/

In [0]:
%sql
--Estadisticas de antiguedad sin valores placeholders
SELECT
  antiguedad,
  COUNT(*) AS cantidad,
  ROUND(MIN(antiguedad),2) AS antiguedad_minimo,
  ROUND(MAX(antiguedad),2) AS antiguedad_maximo,
  ROUND(AVG(antiguedad),2) AS  antiguedad_promedio,
  ROUND(PERCENTILE(antiguedad,0.5),2) AS mediana,
  ROUND(PERCENTILE(antiguedad,0.25),2) AS antiguedad_percentil_25,
  ROUND(PERCENTILE(antiguedad,0.75),2) AS antiguedad_percentil_75
FROM estadisticas_clean
WHERE 
   antiguedad IS NOT NULL 
  AND antiguedad >=0
  AND antiguedad !=999
GROUP BY 
  antiguedad
  
/*
La variable antigüedad está altamente concentrada en 0 años.
Los registros con valores distintos de 0 son muy pocos.
Esto sugiere falta de carga de datos o predominio de propiedades nuevas.
La variable tiene baja utilidad analítica sin limpieza previa.
*/